# Phase 6: Business Policy Simulation
This notebook translates scorecard mathematical probability models into risk-adjusted credit approval decisions. We establish credit risk bands, run credit policy cutoff simulations, and estimate portfolio expected revenue, expected losses, and profit margins.


In [ ]:
import pandas as pd
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning
from scorecard import ScorecardModel
from business_decision_engine import BusinessDecisionEngine
from policy_sim import PolicySimulatorPlotter


## 1. Load Scored Portfolio
We load scored OOT data for credit decision analysis.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
], ['home_ownership', 'purpose'])

train_woe = woe_model.transform(train_df)
oot_woe = woe_model.transform(oot_df)
selected_woe_cols = [f'{col}_woe' for col in woe_model.selected_features]

scorecard = ScorecardModel()
scorecard.fit(train_woe[selected_woe_cols], train_woe['target'])
scorecard.build_scorecard_table(woe_model.mappings)

oot_scores = scorecard.predict_score(oot_woe)
oot_pds = scorecard.predict_pd(oot_woe)

df_scores_pd = oot_df[['loan_amnt', 'target']].copy()
df_scores_pd['score'] = oot_scores
df_scores_pd['pd'] = oot_pds


## 2. Portfolio Risk Bands
Break down the credit portfolio by risk categories: Excellent (750+), Good (700-749), Moderate (650-699), High Risk (600-649), and Very High Risk (<600).


In [ ]:
decision_engine = BusinessDecisionEngine(avg_interest_rate=0.12, loss_given_default=0.60, avg_loan_term_years=3.0)
band_summary = decision_engine.summarize_risk_bands(df_scores_pd)
print(band_summary[['risk_band', 'total_loans', 'total_amount', 'mean_pd', 'expected_revenue', 'expected_loss', 'net_profit']])


## 3. Lending Cutoff Policy Simulation
Simulate portfolio key performance indicators (KPIs) at cutoff scores ranging from 500 to 800.


In [ ]:
cutoff_sim = decision_engine.run_cutoff_simulation(df_scores_pd)
print('Simulation results for selected cutoffs:')
print(cutoff_sim[cutoff_sim['cutoff'].isin([580, 600, 620, 640, 660, 700])])


## 4. Policy Trade-off Curves
Plot Approval Rate vs Expected Portfolio Default Rate and projected Net Profit curves.


In [ ]:
PolicySimulatorPlotter.plot_tradeoff_curves(cutoff_sim)
PolicySimulatorPlotter.plot_profit_optimization(cutoff_sim)


## 5. Executive Lending Guidance
Recommend optimal decision thresholds for management review.


In [ ]:
rec = decision_engine.recommend_optimal_cutoff(cutoff_sim)
print('Optimal Lending Strategies:')
print(f' - Profit-Maximizing: Cutoff {rec["profit_max"]["cutoff"]}, Profit: ${rec["profit_max"]["net_profit"]/1e6:.2f}M, Approval: {rec["profit_max"]["approval_rate"]:.1f}%')
print(f' - Balanced (Low Bad Rate): Cutoff {rec["balanced"]["cutoff"]}, Profit: ${rec["balanced"]["net_profit"]/1e6:.2f}M, Approval: {rec["balanced"]["approval_rate"]:.1f}%')
print(f' - Conservative: Cutoff {rec["conservative"]["cutoff"]}, Profit: ${rec["conservative"]["net_profit"]/1e6:.2f}M, Approval: {rec["conservative"]["approval_rate"]:.1f}%')
